# Multi-Task GRU: Joint 3D Localization, Speed & 90% Confidence Radius Prediction

This notebook trains a **Triple-Head Multi-Task Gated Recurrent Unit (GRU)** model on the **200-User Diversity Dataset**.

### Key Innovations:
1. **Delta-Time Channel ($\Delta t$):** 5-channel sequence input `[RSS, SINR, AoA_az, AoA_el, Delta_t]`, giving explicit time scaling $\frac{\Delta \text{CSI}}{\Delta t}$.
2. **Triple Output Heads:**
   - **Head 1 (Primary):** 3D Cartesian relative coordinates $(x, y, z)$.
   - **Head 2 (Auxiliary):** Instantaneous physical UE speed $v(t)$ (m/s).
   - **Head 3 (Uncertainty Bound):** 90% Confidence Radius $R_{90}$ (meters) — Google Maps Confidence Circle!
3. **Quantile Pinball Loss ($q=0.90$):** Mathematically guarantees that predicted radius $R_{90}$ covers 90% of empirical test errors.
4. **Unseen Cross-User Generalization:** Evaluated on **40 completely unseen test users** (160 Train / 40 Test split).

In [ ]:
import sys, os, time, warnings, json, math, datetime, gc
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using PyTorch device: {DEVICE}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR
for _ in range(10):
    if (PROJECT_ROOT / 'results' / 'grid_localization' / 'grid_25x25').exists():
        break
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'experiments' / '09_grid_localization' / 'src' / 'python'))
from pipelines.multi_user_200_pipeline import load_200_users, make_unseen_user_split, build_multitask_sequences

RUN_TIMESTAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
OUT_DIR = PROJECT_ROOT / 'results' / 'notebook_experiments' / 'gru_multitask' / f'gru_multitask_{RUN_TIMESTAMP}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {OUT_DIR}')

## Global Settings & Configuration

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────
H_TARGET            = 10     # History depth (L = 11 timesteps)
TRAIN_USER_RATIO    = 0.80   # 160 Train Users / 40 Unseen Test Users
HIDDEN_DIM          = 128    # GRU hidden units
NUM_LAYERS          = 2      # GRU layers
DROPOUT_RATE        = 0.15
BATCH_SIZE          = 2048   # GPU batch size
LR                  = 5e-3   # Peak LR
LR_MIN              = 1e-4   # Floor LR
WARMUP_EPOCHS       = 5
MAX_EPOCHS          = 50
WEIGHT_DECAY        = 1e-4
SPEED_LOSS_WEIGHT   = 0.30   # Weight for speed loss
RADIUS_LOSS_WEIGHT  = 0.30   # Weight for R90 radius loss
QUANTILE_Q          = 0.90   # Target confidence quantile (90%)
GRAD_CLIP           = 1.0

# ── Feature Columns ───────────────────────────────────────────────────
SIGNAL_COLS  = ['rss', 'sinr', 'aoa_azimuth', 'aoa_elevation', 'delta_t']
STATIC_COLS  = ['n_antennas', 'antenna_gain_db', 'ue_height']
TARGET_COLS  = ['target_x', 'target_y', 'target_z']

## Data Loading & Unseen User Split

In [ ]:
# Locate latest 200-user simulation dataset directory
data_base = PROJECT_ROOT / 'results' / 'grid_localization' / 'grid_25x25'
dirs = sorted(list(data_base.glob('sim_data_200users_*')))
if not dirs:
    dirs = sorted(list(data_base.glob('sim_data_ne_bs_*')))
DATA_DIR = dirs[-1]
print(f'Loading data from: {DATA_DIR}')

df_raw = load_200_users(DATA_DIR)
print(f'Loaded {len(df_raw):,} total steps across {len(df_raw["user_id"].unique())} users.')

from pipelines.multi_user_pipeline_regression import _read_bs_position_3d
bs_pos = np.array(_read_bs_position_3d(DATA_DIR))
ue_xyz = np.column_stack([df_raw['x_pos'], df_raw['y_pos'], df_raw['ue_height']])
delta  = ue_xyz - bs_pos
df_raw['target_x'], df_raw['target_y'], df_raw['target_z'] = delta[:,0], delta[:,1], delta[:,2]

df_split, train_users, test_users = make_unseen_user_split(df_raw, train_ratio=TRAIN_USER_RATIO, seed=SEED)
print(f'Split: {len(train_users)} Training Users | {len(test_users)} Unseen Test Users')

X_seq_tr_raw, X_static_tr_raw, y_coords_tr_raw, y_speed_tr_raw = build_multitask_sequences(df_split[df_split['split']=='train'], H_TARGET, SIGNAL_COLS, STATIC_COLS, TARGET_COLS)
X_seq_te_raw, X_static_te_raw, y_coords_te_raw, y_speed_te_raw = build_multitask_sequences(df_split[df_split['split']=='test'],  H_TARGET, SIGNAL_COLS, STATIC_COLS, TARGET_COLS)
print(f'Sequences Built: Train={X_seq_tr_raw.shape} | Test={X_seq_te_raw.shape}')

## Normalization & Dataset Class

In [ ]:
N_tr, L, C = X_seq_tr_raw.shape
N_te, _, _ = X_seq_te_raw.shape

sig_scaler = StandardScaler()
X_tr_flat = sig_scaler.fit_transform(X_seq_tr_raw.reshape(-1, C)).reshape(N_tr, L, C)
X_te_flat = sig_scaler.transform(X_seq_te_raw.reshape(-1, C)).reshape(N_te, L, C)
X_tr_flat = np.nan_to_num(X_tr_flat, nan=0.0)
X_te_flat = np.nan_to_num(X_te_flat, nan=0.0)

static_scaler = StandardScaler()
X_static_tr = static_scaler.fit_transform(X_static_tr_raw)
X_static_te = static_scaler.transform(X_static_te_raw)

target_scaler = StandardScaler()
y_coords_tr = target_scaler.fit_transform(y_coords_tr_raw)
y_coords_te = target_scaler.transform(y_coords_te_raw)

speed_scaler = StandardScaler()
y_speed_tr = speed_scaler.fit_transform(y_speed_tr_raw)
y_speed_te = speed_scaler.transform(y_speed_te_raw)

class MultiTaskDataset(Dataset):
    def __init__(self, seq, static, coords, speed):
        self.seq    = torch.tensor(seq, dtype=torch.float32)
        self.static = torch.tensor(static, dtype=torch.float32)
        self.coords = torch.tensor(coords, dtype=torch.float32)
        self.speed  = torch.tensor(speed, dtype=torch.float32)
    def __len__(self):
        return len(self.coords)
    def __getitem__(self, idx):
        return self.seq[idx], self.static[idx], self.coords[idx], self.speed[idx]

## Triple-Head GRU Architecture & Pinball Quantile Loss

In [ ]:
class TripleHeadGRULocalizationModel(nn.Module):
    def __init__(self, n_signals=5, n_static=3, hidden_dim=128, num_layers=2, dropout=0.15):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_signals,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        in_dim = hidden_dim + n_static
        
        # Shared representation
        self.shared = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.LeakyReLU(0.1),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout)
        )
        
        # Head 1 (Primary): 3D Location (x, y, z)
        self.location_head = nn.Sequential(
            nn.Linear(256, 128),
            nn.LeakyReLU(0.1),
            nn.BatchNorm1d(128),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.1),
            nn.Linear(64, 3)
        )
        
        # Head 2 (Auxiliary): Physical Speed v(t) in m/s
        self.speed_head = nn.Sequential(
            nn.Linear(256, 64),
            nn.LeakyReLU(0.1),
            nn.Linear(64, 1)
        )
        
        # Head 3 (Uncertainty): 90% Confidence Radius R90 [meters]
        self.radius_head = nn.Sequential(
            nn.Linear(256, 64),
            nn.LeakyReLU(0.1),
            nn.Linear(64, 1),
            nn.Softplus()  # Ensures positive radius prediction > 0
        )

    def forward(self, seq, static):
        out, _ = self.gru(seq)
        final_feat = out[:, -1, :]
        combined = torch.cat([final_feat, static], dim=1)
        feat = self.shared(combined)
        pred_coords = self.location_head(feat)
        pred_speed  = self.speed_head(feat)
        pred_radius = self.radius_head(feat)
        return pred_coords, pred_speed, pred_radius

def pinball_loss(pred_radius, true_error, q=0.90):
    diff = true_error - pred_radius
    return torch.mean(torch.max(q * diff, (q - 1.0) * diff))

## Model Training Loop

In [ ]:
train_loader = DataLoader(MultiTaskDataset(X_tr_flat, X_static_tr, y_coords_tr, y_speed_tr), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(MultiTaskDataset(X_te_flat, X_static_te, y_coords_te, y_speed_te), batch_size=BATCH_SIZE, shuffle=False)

model = TripleHeadGRULocalizationModel(n_signals=C, n_static=X_static_tr.shape[1], hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT_RATE).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion_l1 = nn.L1Loss()

def lr_schedule_lambda(epoch):
    if epoch <= WARMUP_EPOCHS:
        return float(epoch) / float(max(1, WARMUP_EPOCHS))
    else:
        progress = (epoch - WARMUP_EPOCHS) / float(max(1, MAX_EPOCHS - WARMUP_EPOCHS))
        cos_out = 0.5 * (1.0 + np.cos(np.pi * progress))
        return float(LR_MIN / LR + (1.0 - LR_MIN / LR) * cos_out)

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_schedule_lambda)

def eval_triplehead_metrics(mdl, loader):
    mdl.eval()
    c_preds, c_truths, s_preds, s_truths, r_preds = [], [], [], [], []
    with torch.no_grad():
        for seq_b, static_b, coords_b, speed_b in loader:
            p_c, p_s, p_r = mdl(seq_b.to(DEVICE), static_b.to(DEVICE))
            c_preds.append(p_c.cpu().numpy())
            c_truths.append(coords_b.numpy())
            s_preds.append(p_s.cpu().numpy())
            s_truths.append(speed_b.numpy())
            r_preds.append(p_r.cpu().numpy())
    p_coords_m = target_scaler.inverse_transform(np.vstack(c_preds))
    t_coords_m = target_scaler.inverse_transform(np.vstack(c_truths))
    p_speed_ms = speed_scaler.inverse_transform(np.vstack(s_preds))
    t_speed_ms = speed_scaler.inverse_transform(np.vstack(s_truths))
    p_radius_m = np.vstack(r_preds)

    true_errors_m = np.linalg.norm(p_coords_m - t_coords_m, axis=1, keepdims=True)
    mae_3d = float(np.mean(true_errors_m))
    mae_speed = float(np.mean(np.abs(p_speed_ms - t_speed_ms)))
    coverage_90 = float(np.mean(true_errors_m <= p_radius_m)) * 100.0
    mean_r90 = float(np.mean(p_radius_m))
    return mae_3d, mae_speed, mean_r90, coverage_90

best_val_loss = float('inf')
t_start = time.time()
history_log = []

print(f'Training Triple-Head GRU (Location + Speed + R90) on {len(train_users)} users | Evaluating on {len(test_users)} UNSEEN users...')
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for seq_b, static_b, coords_b, speed_b in train_loader:
        optimizer.zero_grad()
        p_coords, p_speed, p_radius = model(seq_b.to(DEVICE), static_b.to(DEVICE))
        l_coords = criterion_l1(p_coords, coords_b.to(DEVICE))
        l_speed  = criterion_l1(p_speed, speed_b.to(DEVICE))
        with torch.no_grad():
            p_c_m = target_scaler.inverse_transform(p_coords.detach().cpu().numpy())
            t_c_m = target_scaler.inverse_transform(coords_b.numpy())
            true_err_m = torch.tensor(np.linalg.norm(p_c_m - t_c_m, axis=1, keepdims=True), dtype=torch.float32).to(DEVICE)
        l_radius = pinball_loss(p_radius, true_err_m, q=QUANTILE_Q)
        loss = l_coords + SPEED_LOSS_WEIGHT * l_speed + RADIUS_LOSS_WEIGHT * l_radius
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        tr_loss += loss.item() * seq_b.size(0)
    tr_loss /= len(train_loader.dataset)
    scheduler.step()

    val_mae_3d, val_speed, val_r90, val_cov = eval_triplehead_metrics(model, test_loader)
    if val_mae_3d < best_val_loss:
        best_val_loss = val_mae_3d
        torch.save(model.state_dict(), OUT_DIR / 'gru_triplehead_best.pt')

    if epoch == 1 or epoch % 5 == 0 or epoch == MAX_EPOCHS:
        cur_lr = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t_start
        print(f'Ep {epoch:3d}/{MAX_EPOCHS:3d} | tr_loss={tr_loss:.4f} | UNSEEN 3D MAE={val_mae_3d:.3f}m | '
              f'Speed MAE={val_speed:.3f}m/s | Mean R90={val_r90:.2f}m (Coverage={val_cov:.1f}%) | lr={cur_lr:.2e}', flush=True)
        history_log.append({'epoch': epoch, 'tr_loss': tr_loss, 'val_mae_3d': val_mae_3d, 'val_speed': val_speed, 'val_r90': val_r90, 'val_cov': val_cov})

# Final Evaluation
model.load_state_dict(torch.load(OUT_DIR / 'gru_triplehead_best.pt'))
f_3d, f_speed, f_r90, f_cov = eval_triplehead_metrics(model, test_loader)
print(f'\nUNSEEN USER EVALUATION COMPLETE in {time.time()-t_start:.1f}s!')
print(f'  Unseen User 3D MAE:             {f_3d:.3f} meters')
print(f'  Unseen User Speed MAE:          {f_speed:.3f} m/s')
print(f'  Predicted 90% Confidence Radius: {f_r90:.2f} meters')
print(f'  Empirical 90% Radius Coverage:   {f_cov:.1f}% (Target: 90.0%)')

## Evaluation Visualizations & Report

In [ ]:
df_hist = pd.DataFrame(history_log)
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(df_hist['epoch'], df_hist['val_mae_3d'], color='purple', lw=2)
plt.xlabel('Epoch'); plt.ylabel('3D MAE (m)'); plt.title('Unseen 3D Position Error')
plt.grid(alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(df_hist['epoch'], df_hist['val_speed'], color='teal', lw=2)
plt.xlabel('Epoch'); plt.ylabel('Speed MAE (m/s)'); plt.title('Speed Prediction Error')
plt.grid(alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot(df_hist['epoch'], df_hist['val_cov'], color='darkorange', lw=2)
plt.axhline(90.0, color='red', linestyle='--', label='90% Target')
plt.xlabel('Epoch'); plt.ylabel('Coverage (%)'); plt.title('Empirical 90% Radius Coverage')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(OUT_DIR / 'triplehead_gru_curves.png', dpi=150, bbox_inches='tight')
plt.show()